# Content

### Model Summary:
The code implements a text classification model using a **Sequential model** with:
1. **Embedding Layer**: Converts text into dense word vectors (32 dimensions).
2. **LSTM Layer**: Captures sequential context with 64 units.
3. **Dropout Layers**: Reduces overfitting by dropping 50% of neurons.
4. **Dense Layers**: Extracts features and outputs probabilities for 3 categories using softmax.

### Problem Solved:
The model solves the problem of **classifying text into predefined categories** by:
1. Capturing word context with LSTM.
2. Handling class imbalance with computed class weights.
3. Standardizing input length with tokenization and padding.
4. Preventing overfitting using dropout layers.

### Result:
The trained model can categorize unseen sentences into one of three categories, demonstrating accurate pattern recognition and generalization.

# Basic setting

### Measuring execution time

In [ ]:
!pip install --q ipython-autotime
%load_ext autotime

### Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Data collection

### Get data with Labling

In [ ]:
corpus = [
    'This is the first document.',
    'This document is the second document.',
    'And this is the third one.',
    'Is this the first document?',
    'This is a completely new document.',
    'A document is a piece of information.',
    'Is this your first document?',
    'Every document has its own purpose.',
    'The fourth document is quite different.',
    'Here is another unique document.',
    'Do you find this document useful?',
    'Documents can store valuable information.',
    'This is yet another example document.',
    'Some documents are very informative.',
    'Each document serves a specific purpose.',
]

labels = [0, 1, 2, 0, 1, 2, 0, 1, 2, 1, 0, 2, 1, 0, 2]  # Sentence Categories

# Data preparation

### Tokenizer

In [ ]:
vocab_size = 200
maxlen = 12  # Maximum sequence length
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(corpus)
sequences = tokenizer.texts_to_sequences(corpus)

### Pedding and One-Hot Encoding

In [ ]:
# Padding
maxlen = 12
padded = pad_sequences(sequences, maxlen=maxlen, padding='post', truncating='post')

# One-Hot Encoding
num_classes = len(set(labels))  # Number of classes
labels_categorical = to_categorical(labels, num_classes=num_classes)

### Split: train, val

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(padded, labels_categorical, test_size=0.2, random_state=42)

# Model

### Create model

In [ ]:
embedding_dim = 32  # Size of embedding vector

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=maxlen),
    LSTM(64, return_sequences=False),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.summary()

### Complie model

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


### Learning

In [ ]:
# Learning1: Not to apply weight
# model.fit(X_train, y_train, epochs=30, batch_size=8, validation_data=(X_val, y_val), verbose=1)

# Learning2: apply weight
class_weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weights = dict(enumerate(class_weights))
model.fit(X_train, y_train, epochs=40, batch_size=8, validation_data=(X_val, y_val), class_weight=class_weights, verbose=1)

# Evalutate

### Prediction

In [ ]:
new_sentences = [
    'This is a new document.',
    'Is this the second one?',
    'This document is completely new.',
    'Every document has its value.',
    'This is the fourth document and it is unique.',
    'Do you think this document is valuable?',
    'Some documents hold critical information.'
]
new_sequences = tokenizer.texts_to_sequences(new_sentences)
new_padded = pad_sequences(new_sequences, maxlen=maxlen, padding='post', truncating='post')

predictions = model.predict(new_padded)

### Show perfomance

In [ ]:
for i, sentence in enumerate(new_sentences):
    print(f"'{sentence}' -> Category predicted: {np.argmax(predictions[i])}")